# Week 10 · Notebook 2 — Embeddings & a ChromaDB Vector Store
**CSCI 250 · Fall 2026**

Turn text into vectors with **Hugging Face sentence-transformers**, then build and query a **ChromaDB** vector store. No API key required for this notebook.

> Cells degrade gracefully if a package is missing.

## 0. Install

In [ ]:
!pip -q install sentence-transformers chromadb langchain-chroma \
    langchain-huggingface numpy

## 1. Embeddings = meaning as vectors
Texts with similar meaning land close together. We measure closeness with **cosine similarity** (1.0 = identical direction).

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')   # 384-dim, free, local
texts = ['a cat on a mat', 'a kitten on a rug', 'the stock market fell']
vecs = model.encode(texts)
print('shape:', vecs.shape)

def cos(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

print('cat vs kitten:', round(cos(vecs[0], vecs[1]), 3))
print('cat vs market:', round(cos(vecs[0], vecs[2]), 3))

## 2. A ChromaDB vector store (no key)
Chroma auto-embeds your text with a built-in sentence-transformers model, so this runs offline.

In [ ]:
import chromadb
client = chromadb.Client()                 # in-memory
col = client.create_collection('week10')

col.add(
    documents=[
        'LangChain chains LLM steps with the pipe operator.',
        'ChromaDB is an open-source vector store for embeddings.',
        'Embeddings capture the meaning of text as vectors.',
        'LlamaIndex indexes documents for question answering.',
        'Cosine similarity measures how close two vectors are.',
    ],
    ids=['d1', 'd2', 'd3', 'd4', 'd5'],
)
print('stored', col.count(), 'documents')

## 3. Similarity search
Query by meaning, not keywords. Note the query word "database" never appears in the stored text, yet the vector-store result is still correct.

In [ ]:
res = col.query(query_texts=['What database holds vectors?'], n_results=2)
for doc, dist in zip(res['documents'][0], res['distances'][0]):
    print(round(dist, 3), '|', doc)

## 4. Persist to disk
Swap `Client()` for `PersistentClient(path=...)` so your store survives restarts.

In [ ]:
pclient = chromadb.PersistentClient(path='./chroma_db')
pcol = pclient.get_or_create_collection('persisted')
pcol.add(documents=['This survives a kernel restart.'], ids=['p1'])
print('persisted count:', pcol.count())

## 5. LangChain's Chroma wrapper + your own embeddings
This plugs straight into LangChain chains and lets you choose the embedding model explicitly.

In [ ]:
try:
    from langchain_chroma import Chroma
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_core.documents import Document

    emb = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
    docs = [Document(page_content=t) for t in [
        'Chunking splits long text into overlapping pieces.',
        'Retrieval finds the chunks most relevant to a question.',
        'Grounding stuffs retrieved chunks into the prompt.',
    ]]
    store = Chroma.from_documents(docs, embedding=emb)
    for d in store.similarity_search('How do we pick relevant text?', k=2):
        print('-', d.page_content)
except Exception as e:
    print('LangChain Chroma demo skipped:', e)

## 6. Optional: Gemini embeddings instead of local
If you have a Gemini key and want hosted embeddings (still **not** OpenAI), use `GoogleGenerativeAIEmbeddings` with model `text-embedding-004`.

In [ ]:
import os
try:
    if not os.environ.get('GOOGLE_API_KEY') and os.environ.get('GEMINI_API_KEY'):
        os.environ['GOOGLE_API_KEY'] = os.environ['GEMINI_API_KEY']
    from langchain_google_genai import GoogleGenerativeAIEmbeddings
    gemb = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004')
    v = gemb.embed_query('embeddings turn text into vectors')
    print('Gemini embedding length:', len(v))
except Exception as e:
    print('Gemini embeddings skipped (no key / package):', e)

## 7. Recap
You built and queried a real vector store. Next week we add the **generation** step: retrieve relevant chunks, ground a Claude/Gemini prompt in them, and cite the sources — that is **RAG**.